In [1]:
import pandas as pd
import matplotlib.pyplot as plt

selected_series = pd.read_csv(
    "../data/favorita/store44_item589403.csv"
)

selected_series["date"] = pd.to_datetime(selected_series["date"])

selected_series.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,32890,2013-01-02,44,589403,5.0,NaN
1,73313,2013-01-03,44,589403,3.0,NaN
2,113164,2013-01-04,44,589403,8.0,NaN
3,155274,2013-01-05,44,589403,16.0,NaN
4,197250,2013-01-06,44,589403,2.0,NaN


In [2]:
print("Total rows:", len(selected_series))

print("\nPromotion values:")
print(selected_series["onpromotion"].value_counts(dropna=False))

Total rows: 1635

Promotion values:
onpromotion
False    756
NaN      450
True     429
Name: count, dtype: int64


In [ ]:
top_candidates = pd.DataFrame({
    "store_nbr": [44, 3, 7, 51, 49, 48],
    "item_nbr":  [589403, 589403, 589403, 589403, 589403, 346065]
})

top_candidates

In [3]:
top_candidates = pd.DataFrame({
    "store_nbr": [44, 3, 7, 51, 49, 48],
    "item_nbr":  [589403, 589403, 589403, 589403, 589403, 346065]
})

top_candidates

,store_nbr,item_nbr
0,44,589403
1,3,589403
2,7,589403
3,51,589403
4,49,589403
5,48,346065


In [4]:
train_path = "../data/favorita/favorita-grocery-sales-forecasting/train.csv"

candidate_chunks = []

for chunk in pd.read_csv(
    train_path,
    chunksize=1_000_000
):
    # Keep only rows whose store + item combination
    # appears in our top_candidates table
    matching_rows = chunk.merge(
        top_candidates,
        on=["store_nbr", "item_nbr"],
        how="inner"
    )

    if not matching_rows.empty:
        candidate_chunks.append(matching_rows)

all_candidates = pd.concat(
    candidate_chunks,
    ignore_index=True
)

print("Rows extracted:", len(all_candidates))

C:\Users\karra\AppData\Local\Temp\ipykernel_14152\1746213280.py:5: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Rows extracted: 9329


In [5]:
promotion_comparison = (
    all_candidates
    .groupby(["store_nbr", "item_nbr"])["onpromotion"]
    .agg(
        total_rows="size",
        true_count=lambda x: (x == True).sum(),
        false_count=lambda x: (x == False).sum(),
        nan_count=lambda x: x.isna().sum()
    )
    .reset_index()
)

promotion_comparison["promo_percentage"] = (
    promotion_comparison["true_count"]
    / promotion_comparison["total_rows"]
    * 100
)

promotion_comparison

,store_nbr,item_nbr,total_rows,true_count,false_count,nan_count,promo_percentage
0,3,589403,1593,416,739,438,26.114250
1,7,589403,1429,403,651,375,28.201540
2,44,589403,1635,429,756,450,26.238532
3,48,346065,1575,377,753,445,23.936508
4,49,589403,1516,397,707,412,26.187335
5,51,589403,1581,399,759,423,25.237192


In [6]:
sales_comparison = (
    all_candidates
    .groupby(["store_nbr", "item_nbr"])["unit_sales"]
    .agg(
        mean_sales="mean",
        median_sales="median",
        min_sales="min",
        max_sales="max",
        std_sales="std"
    )
    .reset_index()
)

sales_comparison

,store_nbr,item_nbr,mean_sales,median_sales,min_sales,max_sales,std_sales
0,3,589403,6.283114,5.0,1.0,44.0,4.426346
1,7,589403,4.698390,4.0,1.0,34.0,3.568557
2,44,589403,10.894801,10.0,1.0,53.0,6.712577
3,48,346065,15.924444,10.0,1.0,527.0,26.417573
4,49,589403,5.318602,4.0,1.0,36.0,4.104985
5,51,589403,5.650854,5.0,1.0,27.0,3.637296


In [7]:
promotion_sales = (
    all_candidates[
        all_candidates["onpromotion"].notna()
    ]
    .groupby(
        ["store_nbr", "item_nbr", "onpromotion"]
    )["unit_sales"]
    .mean()
    .unstack()
    .reset_index()
)

promotion_sales.columns = [
    "store_nbr",
    "item_nbr",
    "avg_sales_no_promo",
    "avg_sales_promo"
]

promotion_sales

,store_nbr,item_nbr,avg_sales_no_promo,avg_sales_promo
0,3,589403,5.416779,8.497596
1,7,589403,4.076805,6.732010
2,44,589403,10.146825,12.769231
3,48,346065,14.450199,19.984085
4,49,589403,4.586987,6.984887
5,51,589403,5.611331,6.358396


In [9]:
promotion_sales["promo_lift_percentage"] = (
    (
        promotion_sales["avg_sales_promo"]
        - promotion_sales["avg_sales_no_promo"]
    )
    / promotion_sales["avg_sales_no_promo"]
) * 100

promotion_sales

,store_nbr,item_nbr,avg_sales_no_promo,avg_sales_promo,promo_lift_percentage
0,3,589403,5.416779,8.497596,56.875432
1,7,589403,4.076805,6.732010,65.129558
2,44,589403,10.146825,12.769231,25.844590
3,48,346065,14.450199,19.984085,38.296259
4,49,589403,4.586987,6.984887,52.276129
5,51,589403,5.611331,6.358396,13.313514


In [10]:
# Convert the date column from text/string into actual Pandas datetime values
# We need datetime values so Pandas can calculate differences between dates
all_candidates["date"] = pd.to_datetime(all_candidates["date"])


# Group the data by each unique store + item combination
date_comparison = (
    all_candidates
    .groupby(["store_nbr", "item_nbr"])
    .agg(
        # Earliest date available for this store-item combination
        first_date=("date", "min"),

        # Latest date available for this store-item combination
        last_date=("date", "max"),

        # Number of unique dates that actually appear in the dataset
        observed_days=("date", "nunique")
    )
    .reset_index()
)


# Calculate how many calendar days exist between the
# first date and last date, including both endpoints
date_comparison["calendar_days"] = (
    date_comparison["last_date"]
    - date_comparison["first_date"]
).dt.days + 1


# Calculate how many calendar dates are absent from the raw dataset
date_comparison["missing_calendar_days"] = (
    date_comparison["calendar_days"]
    - date_comparison["observed_days"]
)


# Display the final comparison table
date_comparison

,store_nbr,item_nbr,first_date,last_date,observed_days,calendar_days,missing_calendar_days
0,3,589403,2013-01-02,2017-08-15,1593,1687,94
1,7,589403,2013-01-02,2017-08-15,1429,1687,258
2,44,589403,2013-01-02,2017-08-15,1635,1687,52
3,48,346065,2013-01-02,2017-08-15,1575,1687,112
4,49,589403,2013-01-02,2017-08-13,1516,1685,169
5,51,589403,2013-01-02,2017-08-15,1581,1687,106


In [11]:
# Save all six extracted candidate series.
# This means we won't need to scan the original 5 GB train.csv
# again if we want to revisit our candidate analysis later.
all_candidates.to_csv(
    "../data/favorita/top6_candidates.csv",
    index=False
)


# Extract only the final store-item series that we selected.
final_series = all_candidates[
    (all_candidates["store_nbr"] == 3) &
    (all_candidates["item_nbr"] == 589403)
].copy()


# Save our selected series separately.
# Future notebooks can load this small file directly.
final_series.to_csv(
    "../data/favorita/store3_item589403.csv",
    index=False
)


# Quick confirmation
print("Final series rows:", len(final_series))
print("First date:", final_series["date"].min())
print("Last date:", final_series["date"].max())

Final series rows: 1593
First date: 2013-01-02 00:00:00
Last date: 2017-08-15 00:00:00
